In [ ]:
import json
from gurobipy import Model, GRB, quicksum

# 1. Carica dati da JSON
with open("input_data.json") as f:
    D = json.load(f)

n     = D["n_attivita"]
dur   = D["durata_min"]
mac   = D["macchinari_richiesti"]
consM = D["consumo_macchinari"]
P     = D["precedenze"]
TW    = D["time_windows_min"]
fasce = D["fasce_orarie"]
c_f   = D["costi_fasce"]
e_req = D["impiegati_richiesti"]
e_avl = D["impiegati_disponibili"]
Delta = D["Delta_min"]
H     = D["H"]

I = range(n)
M = range(len(consM))

# Matrice binaria R[(i,m)]
R = {(i,m):1 for i in I for m in mac[i]}

# 2. Costruisci modello
model = Model("MinPeakPower")

# 3. Variabili
x = model.addVars(I, range(H), vtype=GRB.BINARY, name="x")
y = model.addVars(I, range(H), vtype=GRB.BINARY, name="y")
z = model.addVars(I, range(H), vtype=GRB.BINARY, name="z_emp")  # per impiegati
v = model.addVar(vtype=GRB.CONTINUOUS, name="Pmax")

# 4. Parametri ausiliari
d_slots = {i: dur[i] // Delta for i in I}

# 5. Vincoli

# 5.1 Ogni attività parte esattamente una volta
for i in I:
    model.addConstr(quicksum(x[i,l] for l in range(H - d_slots[i] + 1)) == 1)

# 5.2 Definizione y[i,l]
for i in I:
    for l in range(H):
        model.addConstr(
            y[i,l] == quicksum(
                x[i,t] 
                for t in range(max(0, l-d_slots[i]+1), 
                               min(l+1, H-d_slots[i]+1))
            )
        )

# 5.3 Non-sovrapp. macchinari
for m in M:
    for l in range(H):
        model.addConstr(
            quicksum(y[i,l]*R.get((i,m),0) for i in I) <= 1
        )

# 5.4 Precedenze
for i,j in P:
    model.addConstr(
        quicksum(l*x[i,l] for l in range(H-d_slots[i]+1)) + d_slots[i]
        <=
        quicksum(l*x[j,l] for l in range(H-d_slots[j]+1))
    )

# 5.5 Finestra oraria
for i in I:
    a,b = TW[i]
    a_slot = (a - D["inizio_giornata_min"])//Delta
    b_slot = (b - D["inizio_giornata_min"])//Delta
    for l in range(H):
        if l < a_slot or l > b_slot - d_slots[i]:
            model.addConstr(x[i,l]==0)

# 5.6 Vincolo impiegati
for l in range(H):
    model.addConstr(
        quicksum(e_req[i]*y[i,l] for i in I) <= e_avl[l]
    )

# 5.7 Vincolo picco potenza
for l in range(H):
    model.addConstr(
        quicksum(y[i,l]*sum(R.get((i,m),0)*consM[m] for m in M) for i in I) <= v
    )

# 6. Obiettivo
model.setObjective(v, GRB.MINIMIZE)

# 7. Risolvi
model.optimize()

# 8. Stampa risultato
if model.status == GRB.OPTIMAL:
    print("Picco =", v.X, "kW")
    for i in I:
        for l in range(H):
            if x[i,l].X > 0.5:
                t_min = D["inizio_giornata_min"] + l*Delta
                print(f"Attività {i} inizia alle {t_min//60:02d}:{t_min%60:02d}")
else:
    print("Problema infeasible o non ottimale.")
